#  Multimodal Image Captioning & Text Fusion Framework

An advanced, end-to-end computer vision and natural language processing pipeline designed to generate, rank, and intelligently fuse image descriptions. This project leverages an ensemble of foundational models to eliminate factual hallucinations and capture fine-grained scene semantics.

---

###  Why the Flickr8k Dataset is Used

* **Gold Standard for Multi-Hypothesis Evaluation:** Every single image contains exactly **5 distinct, human-annotated reference captions**. This unique structure makes it the perfect dataset to test consensus algorithms, majority voting, and text-fusion layers.
* **Rich Variety of Diverse Descriptions:** The 5 human captions describe the same scene from different perspectives—some focus on global layout (e.g., *"a wooden building"*), while others zoom in on fine-grained visual details (e.g., *"a pink dress"*). This diversity provides excellent raw material for **Qwen/Llama** to perform intelligent synthesis.
* **Computationally Lightweight & Portable:** Comprising **8,092 images** in total, it provides enough statistical variance to calculate meaningful validation metrics (**BLEU, METEOR, CIDEr**) without running into the massive storage or CPU/GPU memory constraints of larger datasets like MS-COCO.

---
###  Key Capabilities & Architecture
* **Candidate Generation:** Generates diverse description hypotheses using **Florence-2** via beam search.
* **Dual-Metric Evaluation:** Scores text alignment with cross-modal checkpoints using local **BLIP ITM** matching layers alongside a cloud-routed semantic **Jina AI Reranker**.
* **Intelligent Synthesis:** Executes a majority-voting consensus filter to feed optimal structural contextual prompts into a **Qwen-2.5-1.5B / Llama 3.2** text-fusion layer.

In [ ]:
# Importing libraries
import os
import torch
import re
import gc
import ast
import string
import numpy as np
import pandas as pd
from PIL import Image
from collections import Counter
import matplotlib.gridspec as gridspec
from transformers import BlipProcessor, BlipForConditionalGeneration, BlipModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from nltk.translate.meteor_score import meteor_score
import matplotlib.pyplot as plt
from collections import defaultdict

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMAGES_PATH   = "/content/drive/MyDrive/flickr8k/Images"
CAPTIONS_PATH = "/content/drive/MyDrive/flickr8k/Flickr_TextData/Flickr8k.token.txt"
SAVE_PATH     = "/content/drive/MyDrive/flickr8k/flickr8k_embeddings.npz"
SAVE_PATH_IMG = "/content/drive/MyDrive/flickr8k/image_embeddings.npz"
SAVE_PATH_TXT = "/content/drive/MyDrive/flickr8k/caption_embeddings.npz"

print("Device:", DEVICE)

In [ ]:
# 1. Define paths
IMAGES_PATH = "/content/drive/MyDrive/flickr8k/Images"
CAPTIONS_PATH = "/content/drive/MyDrive/flickr8k/Flickr_TextData/Flickr8k.token.txt"

# 2. Parse the captions file and group them by image name
image_to_captions = defaultdict(list)

with open(CAPTIONS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split('\t')
        if len(parts) < 2:
            continue

        img_id, caption = parts[0], parts[1]
        img_name = img_id.split('#')[0]
        image_to_captions[img_name].append(caption)

# 3. Get the first 2 images
sample_images = list(image_to_captions.keys())[:2]

# 4. Create a side-by-side plot layout (1 row, 2 columns)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))  # Adjusted aspect ratio for horizontal space

for i, img_name in enumerate(sample_images):
    full_image_path = os.path.join(IMAGES_PATH, img_name)

    if os.path.exists(full_image_path):
        img = Image.open(full_image_path)

        # Display image in its respective subplot column
        axes[i].imshow(img)
        axes[i].axis('off')
        axes[i].set_title(f"Example {i+1}: {img_name}", fontsize=11, fontweight='bold')
    else:
        axes[i].text(0.5, 0.5, f"Image not found:\n{img_name}",
                     ha='center', va='center', fontsize=10, color='red')
        axes[i].axis('off')

plt.tight_layout()
plt.show()

# 5. Print the captions side-by-side or stacked cleanly beneath the visual block
print("="*45 + " DATASET SAMPLES " + "="*45)
col1_captions = image_to_captions[sample_images[0]]
col2_captions = image_to_captions[sample_images[1]]

# Format text columns neatly using string spacing
print(f"{' Captions for ' + sample_images[0]:<52} |  Captions for {sample_images[1]}")
print("-" * 105)

for idx in range(5):
    cap1 = col1_captions[idx] if idx < len(col1_captions) else ""
    cap2 = col2_captions[idx] if idx < len(col2_captions) else ""

    # Clip long text lines slightly if needed to prevent terminal wrapping
    print(f"  {idx + 1}. {cap1:<48} |   {idx + 1}. {cap2}")
print("="*105)

In [ ]:
captions_data = []
with open(CAPTIONS_PATH, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split('\t')
        if len(parts) == 2:
            img_caption_id, caption = parts
            img_name = img_caption_id.split('#')[0]
            captions_data.append({
                'image_name': img_name,
                'caption': caption
            })

df = pd.DataFrame(captions_data)

captions_dict = df.groupby('image_name')['caption'].apply(list).to_dict()
all_image_names = list(captions_dict.keys())

print("Total rows loaded    :", len(df))
print("Unique images        :", len(all_image_names))
print("Captions per image   :", len(df) // len(all_image_names))
print("\nSample image         :", all_image_names[0])
print("Its captions         :", captions_dict[all_image_names[0]])

In [ ]:
# Build captions_dict
# Format: { 'image1.jpg': ['caption1', 'caption2', ...], ... }

# --- Parse Flickr8k.token.txt → captions_dict ---
captions_dict = {}

with open(CAPTIONS_PATH, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split('\t')
        if len(parts) == 2:
            img_id, caption = parts
            img_name = img_id.split('#')[0]      # 'image1.jpg#0' → 'image1.jpg'

            if img_name not in captions_dict:
                captions_dict[img_name] = []
            captions_dict[img_name].append(caption)

print(f" Total unique images : {len(captions_dict)}")
print(f" Captions per image  : {len(list(captions_dict.values())[0])}")
print(f"\nSample entry:")
sample_key = list(captions_dict.keys())[0]
print(f"  Image : {sample_key}")
for i, cap in enumerate(captions_dict[sample_key]):
    print(f"  Cap {i} : {cap}")

**EDA**

In [ ]:
# Dataset Overview

print(" Dataset Overview")
print(f"  Total rows        : {len(df)}")
print(f"  Total images      : {df['image_name'].nunique()}")
print(f"  Captions per image: {len(df) // df['image_name'].nunique()}")
print(f"  Missing values    : {df.isnull().sum().sum()}")
print(df.head())

In [ ]:
# Distribution of Caption Length vs Frequency
# Fixed for Flickr8k.token.txt format

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

CAPTIONS_PATH  = "/content/drive/MyDrive/flickr8k/Flickr_TextData/Flickr8k.token.txt"

# --- Correct way to load Flickr8k.token.txt ---
# Format per line: 'image.jpg#0\tcaption text here'
rows = []
with open(CAPTIONS_PATH, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split('\t')
        if len(parts) == 2:
            img_name = parts[0].split('#')[0]   # 'image.jpg#0' → 'image.jpg'
            caption  = parts[1]
            rows.append({'image_name': img_name, 'caption': caption})

df = pd.DataFrame(rows)

# --- Caption length ---
df['length'] = df['caption'].apply(lambda x: len(str(x).split()))

# --- Plot ---
plt.figure(figsize=(10, 5))
sns.histplot(df['length'], bins=20, kde=True, color='skyblue')
plt.title('Distribution of Caption Lengths - Flickr8k')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.show()

print(f"Total captions   : {len(df)}")
print(f"Unique images    : {df['image_name'].nunique()}")
print(f"Average length   : {df['length'].mean():.2f} words")
print(f"Max length       : {df['length'].max()} words")
print(f"Min length       : {df['length'].min()} words")
print(f"\nSample rows:")
print(df.head(6))

In [ ]:
# Vocabulary Stats

# Caption length
df['caption_length'] = df['caption'].apply(lambda x: len(str(x).split()))

# Fix: define all_words first, then unique_words from it
all_words    = " ".join(df['caption']).lower().split()
unique_words = set(all_words)

print(f"\nVocabulary Stats:")
print(f"  Total captions      : {len(df)}")
print(f"  Unique images       : {df['image_name'].nunique()}")
print(f"  Total words         : {len(all_words)}")
print(f"  Unique vocabulary   : {len(unique_words)}")
print(f"  Avg caption length  : {df['caption_length'].mean():.1f} words")
print(f"  Min caption length  : {df['caption_length'].min()} words")
print(f"  Max caption length  : {df['caption_length'].max()} words")

In [ ]:
# Distortion in width and height in images


widths, heights = [], []

for img_name in list(captions_dict.keys())[:500]:   # sample 500 images
    img_path = os.path.join(IMAGES_PATH, img_name)  # ← IMAGES_DIR → IMAGES_PATH
    if not os.path.exists(img_path):                 # skip if missing
        continue
    img = Image.open(img_path)
    w, h = img.size
    widths.append(w)
    heights.append(h)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(widths,  bins=20, color='#4C72B0', edgecolor='white')
axes[0].set_title("Image Width Distribution",  fontweight='bold')
axes[0].set_xlabel("Width (pixels)")

axes[1].hist(heights, bins=20, color='#DD8452', edgecolor='white')
axes[1].set_title("Image Height Distribution", fontweight='bold')
axes[1].set_xlabel("Height (pixels)")

plt.tight_layout()
plt.show()

print(f"  Images checked : {len(widths)}")
print(f"  Avg Width      : {sum(widths)/len(widths):.0f}px")
print(f"  Avg Height     : {sum(heights)/len(heights):.0f}px")

In [ ]:
# Top 20 most used words


all_words = " ".join(df['caption']).lower().split()
word_freq  = Counter(all_words)

top_words      = word_freq.most_common(20)
words, counts  = zip(*top_words)

plt.figure(figsize=(12, 4))
plt.bar(words, counts, color='#4C72B0', edgecolor='white')
plt.title("Top 20 Most Common Words in Captions", fontweight='bold')
plt.xlabel("Word")
plt.ylabel("Frequency")
plt.xticks(rotation=45)
for i, count in enumerate(counts):
    plt.text(i, count + 10, str(count), ha='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Frequency and total punctuation characters

from collections import Counter
import re

# ── Find all punctuation characters ──
all_puncts = []
for caption in df['caption']:
    puncts = re.findall(r'[^\w\s]', caption)
    all_puncts.extend(puncts)

punct_freq = Counter(all_puncts)

print("  Punctuation Frequency Breakdown:")
print(f"  Total punctuation characters : {len(all_puncts)}")
print(f"  Unique punctuation types     : {len(punct_freq)}")
print(f"\n  Character  Count   Meaning")
print(f"  {'─'*35}")
for char, count in punct_freq.most_common():
    meanings = {
        '.': 'Full stop / Period',
        ',': 'Comma',
        "'": 'Apostrophe',
        '-': 'Hyphen',
        '"': 'Quotation mark',
        '!': 'Exclamation mark',
        '?': 'Question mark',
        '/': 'Slash',
        ':': 'Colon',
        ';': 'Semicolon',
        '(': 'Open bracket',
        ')': 'Close bracket',
    }
    meaning = meanings.get(char, 'Other')
    print(f"  '{char}'  →  {count:>6}   {meaning}")

In [ ]:
chars  = [char for char, _ in punct_freq.most_common()]
counts = [count for _, count in punct_freq.most_common()]

plt.figure(figsize=(10, 4))
plt.bar(chars, counts, color='gray', edgecolor='white')
plt.title("Punctuation Distribution in Flickr8k Captions", fontweight='bold')
plt.xlabel("Punctuation Character")
plt.ylabel("Frequency")
for i, (char, count) in enumerate(zip(chars, counts)):
    plt.text(i, count + 10, str(count), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Count Emojis

emoji_pattern = re.compile("["
    u"\U0001F600-\U0001F64F"  # emoticons
    u"\U0001F300-\U0001F5FF"  # symbols & pictographs
    u"\U0001F680-\U0001F6FF"  # transport & map
    u"\U0001F1E0-\U0001F1FF"  # flags
    u"\U00002700-\U000027BF"  # dingbats
    u"\U0001F900-\U0001F9FF"  # supplemental symbols
"]+", flags=re.UNICODE)

df['emoji_count'] = df['caption'].apply(
    lambda x: len(emoji_pattern.findall(x))
)

total_emojis = df['emoji_count'].sum()
has_emojis   = (df['emoji_count'] > 0).sum()

print("\n Emoji Stats:")
print(f"  Total emojis found       : {total_emojis}")
print(f"  Captions with emojis     : {has_emojis} / {len(df)}")

**Preprocessing of Dataset**

In [ ]:

def clean_caption(text):
    text = str(text).lower().strip()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['caption_clean'] = df['caption'].apply(clean_caption)

print("caption_clean created")
print(f"\n  Raw      : {df['caption'].iloc[0]}")
print(f"  Cleaned  : {df['caption_clean'].iloc[0]}")

In [ ]:
print("Current columns:", df.columns.tolist())
print("Sample caption :", df['caption_clean'].iloc[0])

BLIP's processor handles resizing internally and automatically. Unlike Gemini embeddings where we had to manually standardize, BLIP's BlipProcessor resizes every image to 384x384 on its own before passing to the model.


In [ ]:

VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png')

valid_images   = []
invalid_images = []

for img_name in df['image_name'].unique():
    if not img_name.lower().endswith(VALID_EXTENSIONS):
        invalid_images.append(img_name)
        continue
    if not os.path.exists(os.path.join(IMAGES_PATH, img_name)):
        invalid_images.append(img_name)
        continue
    valid_images.append(img_name)

df = df[df['image_name'].isin(valid_images)].reset_index(drop=True)

print("Valid images    :", len(valid_images))
print("Invalid/missing :", len(invalid_images))
print("Clean df size   :", len(df), "rows")
print("Columns         :", df.columns.tolist())

In [ ]:
#checking if the baseline human annotations are accurate or not.

import random

sample_imgs = random.sample(list(captions_dict.keys()), 8)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle("\n""Random Sample Images from Flickr8k", fontsize=14, fontweight='bold')

for i, img_name in enumerate(sample_imgs):
    row, col = i // 4, i % 4
    img      = Image.open(os.path.join(IMAGES_PATH, img_name)).convert("RGB")
    axes[row][col].imshow(img)
    axes[row][col].axis('off')
    axes[row][col].set_title(captions_dict[img_name][0][:60], fontsize=7, wrap=True)

plt.tight_layout()
plt.show()

**Feature Extraction**

In [ ]:
import torch
import numpy as np
from PIL import Image
import os
from transformers import BlipProcessor, BlipForConditionalGeneration, BlipForImageTextRetrieval
from sklearn.preprocessing import normalize

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

CAPTION_MODEL_NAME = "Salesforce/blip-image-captioning-large"
ITM_MODEL_NAME     = "Salesforce/blip-itm-large-coco"

print("Loading processor...")
processor = BlipProcessor.from_pretrained(CAPTION_MODEL_NAME)
print("Processor loaded")

print("Loading captioning model...")
caption_model = BlipForConditionalGeneration.from_pretrained(CAPTION_MODEL_NAME).to(DEVICE)
caption_model.eval()
print("Captioning model loaded")

print("Loading ITM model...")
itm_model = BlipForImageTextRetrieval.from_pretrained(ITM_MODEL_NAME).to(DEVICE)
itm_model.eval()
print("ITM model loaded")

print("\nCaption model on :", next(caption_model.parameters()).device)
print("ITM model on     :", next(itm_model.parameters()).device)

**Qwen2-VL-2B**

In [ ]:
'''import subprocess
subprocess.run(["pip", "install", "transformers", "accelerate", "qwen-vl-utils", "-q"])'''

import torch, gc
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"

print("Device :", DEVICE)
print("VRAM   :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

gc.collect()
torch.cuda.empty_cache()

print("\nLoading Qwen2-VL processor...")
vl_processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    trust_remote_code = True
)

print("Loading Qwen2-VL-2B model...")
vl_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype       = torch.float16,
    device_map        = "auto",
    trust_remote_code = True
)
vl_model.eval()

print("\n Qwen2-VL-2B loaded!")
print("VRAM used :", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

# ── Sanity test ──
print("\nRunning sanity test...")
from PIL import Image
import requests
from io import BytesIO

test_img  = Image.new("RGB", (224, 224), color=(100, 150, 200))
messages  = [{
    "role": "user",
    "content": [
        {"type": "image", "image": test_img},
        {"type": "text",  "text": "What color is this image?"}
    ]
}]
text   = vl_processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = vl_processor(
    text=[text], images=[test_img], return_tensors="pt"
).to(DEVICE)

with torch.no_grad():
    out = vl_model.generate(
        **inputs, max_new_tokens=20,
        do_sample=False,
        pad_token_id=vl_processor.tokenizer.eos_token_id
    )
generated = out[0][inputs['input_ids'].shape[1]:]
response  = vl_processor.tokenizer.decode(generated, skip_special_tokens=True)
print(f"Sanity response : {response}")
print(" Sanity test passed!")

In [ ]:
# 5 diverse prompts — each asks from different angle
CAPTION_PROMPTS = [
    "Describe this image in one detailed sentence.",
    "What is happening in this image? Write one descriptive sentence.",
    "Describe the main subjects, actions and setting in one sentence.",
    "Write a detailed caption focusing on people, animals and objects visible.",
    "Describe this scene including background details and activities shown.",
]
TEMPERATURES = [None, None, 0.7, 0.8, 0.9]   # first 2 deterministic

def generate_vl_caption(image, prompt, temperature=None):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": prompt}
        ]
    }]
    text   = vl_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = vl_processor(
        text=[text], images=[image], return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        if temperature:
            out = vl_model.generate(
                **inputs,
                max_new_tokens = 60,
                do_sample      = True,
                temperature    = temperature,
                pad_token_id   = vl_processor.tokenizer.eos_token_id
            )
        else:
            out = vl_model.generate(
                **inputs,
                max_new_tokens = 60,
                do_sample      = False,
                num_beams      = 3,
                pad_token_id   = vl_processor.tokenizer.eos_token_id
            )

    generated = out[0][inputs['input_ids'].shape[1]:]
    return vl_processor.tokenizer.decode(
        generated, skip_special_tokens=True
    ).strip().lower()


def generate_5_captions(image: Image.Image) -> list:
    captions = []
    for prompt, temp in zip(CAPTION_PROMPTS, TEMPERATURES):
        cap = generate_vl_caption(image, prompt, temp)
        captions.append(cap)
        print(f"  ✔ {cap[:80]}")

    # Deduplicate
    seen, unique = set(), []
    for c in captions:
        if c not in seen:
            seen.add(c)
            unique.append(c)
    while len(unique) < 5:
        unique.append(unique[0])

    return unique

print("generate_5_captions defined — ready!")

In [ ]:
IMG_PATH = "/content/drive/MyDrive/flickr8k/Images/1000268201_693b08cb0e.jpg"

test_img = Image.open(IMG_PATH).convert("RGB")

print("=" * 55)
print("  Qwen2-VL-2B — Caption Generation Test")
print("=" * 55)
print(f"  Image : 1000268201_693b08cb0e.jpg\n")
print("Generating 5 captions...\n")

captions = generate_5_captions(test_img)

print("\n" + "=" * 55)
print("  Final 5 Captions:")
print("=" * 55)
for i, c in enumerate(captions):
    print(f"  {i+1}. {c}")

# Show image alongside captions
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(test_img)
axes[0].axis("off")
axes[0].set_title("Input Image", fontsize=11, fontweight='bold')

axes[1].axis("off")
for i, c in enumerate(captions):
    axes[1].text(
        0.02, 0.92 - i * 0.18,
        f"{i+1}. {c}",
        transform  = axes[1].transAxes,
        fontsize   = 9,
        wrap       = True,
        bbox       = dict(boxstyle='round,pad=0.3',
                          facecolor='#eaf4fb',
                          edgecolor='#2980b9',
                          linewidth=1)
    )

axes[1].set_title("Qwen2-VL Generated Captions", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
def generate_5_captions(image: Image.Image) -> list:
    inputs   = processor(images=image, return_tensors="pt").to(DEVICE)
    captions = []

    with torch.no_grad():

        # Caption 1 — greedy beam search
        out = caption_model.generate(
            **inputs, num_beams=5,
            max_new_tokens=40, min_length=8
        )
        captions.append(processor.decode(out[0], skip_special_tokens=True))

        # Caption 2 — longer with length penalty
        out = caption_model.generate(
            **inputs, num_beams=5,
            max_new_tokens=55, min_length=12,
            length_penalty=2.0
        )
        captions.append(processor.decode(out[0], skip_special_tokens=True))

        # Caption 3 — nucleus sampling temp 0.7
        out = caption_model.generate(
            **inputs, do_sample=True,
            temperature=0.7, top_p=0.9,
            max_new_tokens=40, min_length=8
        )
        captions.append(processor.decode(out[0], skip_special_tokens=True))

        # Caption 4 — nucleus sampling temp 0.9
        out = caption_model.generate(
            **inputs, do_sample=True,
            temperature=0.9, top_p=0.95,
            max_new_tokens=45, min_length=8
        )
        captions.append(processor.decode(out[0], skip_special_tokens=True))

        # Caption 5 — top-k sampling
        out = caption_model.generate(
            **inputs, do_sample=True,
            top_k=50, temperature=0.8,
            max_new_tokens=40, min_length=8
        )
        captions.append(processor.decode(out[0], skip_special_tokens=True))

    # Remove duplicates
    seen, unique = set(), []
    for cap in captions:
        cap = cap.strip().lower()
        if cap not in seen:
            seen.add(cap)
            unique.append(cap)
    while len(unique) < 5:
        unique.append(unique[0])

    return unique

# ── Test on one Flickr8k image ──
IMAGES_PATH = "/kaggle/input/datasets/afshaanjum001/flickr8k-images/Images"
fused_df    = pd.read_csv("/kaggle/input/datasets/afshaanjum001/fused-caption/qwen_fused_captions.csv")

test_img    = Image.open(
    os.path.join(IMAGES_PATH, fused_df['image_name'].iloc[0])
).convert("RGB")

print(f"Testing on : {fused_df['image_name'].iloc[0]}\n")
captions = generate_5_captions(test_img)

print("\n 5 Generated Captions:")
for i, c in enumerate(captions):
    print(f"  {i+1}. {c}")

**Image Embeddings**

In [ ]:
# Assuming your images are zipped on drive
!cp /content/drive/MyDrive/path/to/flickr8k.zip /content/
!unzip -q /content/flickr8k.zip -d /content/images_local/
IMAGES_PATH = "/content/images_local/"

In [ ]:
# 1. SETUP: Define Paths & Parameters
# Ensure these match your Drive structure exactly
IMAGES_PATH        = "/content/drive/MyDrive/flickr8k/Images"
SAVE_PATH_IMG      = "/content/drive/MyDrive/flickr8k/image_embeddings.npz"
SAVE_BASE_IMG_TEMP = "/content/drive/MyDrive/flickr8k/image_embeddings_temp"
BATCH_SIZE         = 32

# 2. INITIALIZE: Resume or Start Fresh
# unique_images should be from your dataframe: unique_images = df['image_name'].unique().tolist()
unique_images = df['image_name'].unique().tolist()
total = len(unique_images)

img_emb_list = []
img_name_list = []
already_done = set()

if os.path.exists(SAVE_PATH_IMG):
    try:
        checkpoint = np.load(SAVE_PATH_IMG, allow_pickle=True)
        img_emb_list = list(checkpoint['embeddings'])
        img_name_list = list(checkpoint['image_names'])
        already_done = set(img_name_list)
        print(f"Resuming: {len(already_done)} images already processed.")
    except Exception as e:
        print(f"Starting fresh (checkpoint error): {e}")

# 3. CREATE BATCHES
remaining_images = [img for img in unique_images if img not in already_done]
batches = [remaining_images[i:i + BATCH_SIZE] for i in range(0, len(remaining_images), BATCH_SIZE)]

print(f"Total: {total} | Remaining: {len(remaining_images)} | Batches left: {len(batches)}")

# 4. THE LOOP
for batch_idx, batch in enumerate(batches):
    images_pil = []
    valid_names = []

    for img_name in batch:
        img_path = os.path.join(IMAGES_PATH, img_name)
        if not os.path.exists(img_path):
            continue
        try:
            image = Image.open(img_path)
            image.draft('RGB', (224, 224))
            images_pil.append(image.convert("RGB"))
            valid_names.append(img_name)
        except Exception as e:
            print(f"Skipped {img_name}: {e}")
            continue

    if not images_pil:
        continue

    # Inference
    inputs = processor(images=images_pil, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        # Using the BLIP-Large vision model and projection layer
        vision_out = itm_model.vision_model(pixel_values=inputs["pixel_values"])
        embeddings = itm_model.vision_proj(vision_out.last_hidden_state[:, 0, :])

    img_emb_list.extend(embeddings.cpu().numpy())
    img_name_list.extend(valid_names)

    # Cleanup Memory
    del inputs, vision_out, images_pil
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    gc.collect()

    # Progress & Checkpointing
    if (batch_idx + 1) % 5 == 0:
        print(f"Progress: {len(img_name_list)}/{total}")

    processed_so_far = (batch_idx + 1) * BATCH_SIZE
    if processed_so_far % 160 < BATCH_SIZE:
        np.savez_compressed(
            SAVE_BASE_IMG_TEMP,
            image_names=np.array(img_name_list),
            embeddings=np.array(img_emb_list)
        )
        if os.path.exists(SAVE_PATH_IMG):
            os.remove(SAVE_PATH_IMG)
        os.rename(SAVE_BASE_IMG_TEMP + ".npz", SAVE_PATH_IMG)
        print(f"--- Checkpoint saved at {len(img_name_list)} images ---")

In [ ]:
img_data      = np.load(SAVE_PATH_IMG, allow_pickle=True)
img_names     = img_data['image_names']
img_embeddings= img_data['embeddings']

print("Total saved        :", len(img_names))
print("Unique names       :", len(set(img_names)))
print("Embedding shape    :", img_embeddings.shape)
print("Duplicates present :", len(img_names) - len(set(img_names)))

In [ ]:
IMAGES_PATH = "/content/drive/MyDrive/flickr8k/Images"

if os.path.exists(IMAGES_PATH):
    # List only files with image extensions to be precise
    valid_extensions = ('.jpg', '.jpeg', '.png')
    all_files = os.listdir(IMAGES_PATH)
    image_files = [f for f in all_files if f.lower().endswith(valid_extensions)]

    print(f"Total files in folder   : {len(all_files)}")
    print(f"Total valid image files : {len(image_files)}")

    if len(image_files) < 8000:
        print(" Warning: Your dataset appears incomplete. Expected ~8091 images.")
    else:
        print(" Dataset looks complete!")
else:
    print("Error: IMAGES_PATH does not exist. Check your Drive mount.")

In [ ]:
# Debug cell - run only first 5 batches to check errors
img_data       = np.load(SAVE_PATH_IMG, allow_pickle=True)
img_names_raw  = img_data['image_names']
img_embs_raw   = img_data['embeddings']

print("Before dedup :", len(img_names_raw))

seen      = set()
keep_idx  = []
for i, name in enumerate(img_names_raw):
    if name not in seen:
        seen.add(name)
        keep_idx.append(i)

clean_names = img_names_raw[keep_idx]
clean_embs  = img_embs_raw[keep_idx]

print("After dedup  :", len(clean_names))

SAVE_BASE_IMG      = SAVE_PATH_IMG.replace(".npz", "")
SAVE_BASE_IMG_TEMP = SAVE_BASE_IMG + "_temp"

np.savez_compressed(
    SAVE_BASE_IMG_TEMP,
    image_names = clean_names,
    embeddings  = clean_embs
)
if os.path.exists(SAVE_PATH_IMG):
    os.remove(SAVE_PATH_IMG)
os.rename(SAVE_BASE_IMG_TEMP + ".npz", SAVE_PATH_IMG)

print("Saved clean embeddings. Shape :", clean_embs.shape)
print("Expected                      : (8091, 256)")

In [ ]:
img_data       = np.load(SAVE_PATH_IMG, allow_pickle=True)
img_embeddings = img_data['embeddings']
img_names      = img_data['image_names']

print("Image embeddings shape :", img_embeddings.shape)
print("Unique images          :", len(set(img_names)))

These 28 were lost during the crash and resume cycle. We need to find exactly which ones and embed only those.

In [ ]:
already_done  = set(img_names)
all_images    = set(df['image_name'].unique())
missing_images = list(all_images - already_done)

print("Total in dataset  :", len(all_images))
print("Already embedded  :", len(already_done))
print("Missing           :", len(missing_images))
print("\nMissing images:")
for m in missing_images:
    print(" ", m)

In [ ]:
missing_emb_list  = []
missing_name_list = []

for img_name in missing_images:
    img_path = os.path.join(IMAGES_PATH, img_name)
    try:
        image  = Image.open(img_path).convert("RGB")
        inputs = processor(
            images         = [image],
            return_tensors = "pt",
            padding        = True
        ).to(DEVICE)

        with torch.no_grad():
            vision_out = itm_model.vision_model(pixel_values=inputs["pixel_values"])
            image_feat = itm_model.vision_proj(vision_out.last_hidden_state[:, 0, :])
            embedding  = image_feat.cpu().numpy()
            embedding  = normalize(embedding, norm="l2")

        missing_emb_list.append(embedding[0])
        missing_name_list.append(img_name)
        print(f"Done : {img_name}")

    except Exception as e:
        print(f"Skipped {img_name}: {e}")

print(f"\nMissing images embedded : {len(missing_name_list)}")

In [ ]:
final_names = np.concatenate([img_names, np.array(missing_name_list)])
final_embs  = np.concatenate([img_embeddings, np.array(missing_emb_list)])

print("Final shape :", final_embs.shape)
print("Expected    : (8091, 256)")

SAVE_BASE_IMG      = SAVE_PATH_IMG.replace(".npz", "")
SAVE_BASE_IMG_TEMP = SAVE_BASE_IMG + "_temp"

np.savez_compressed(
    SAVE_BASE_IMG_TEMP,
    image_names = final_names,
    embeddings  = final_embs
)
if os.path.exists(SAVE_PATH_IMG):
    os.remove(SAVE_PATH_IMG)
os.rename(SAVE_BASE_IMG_TEMP + ".npz", SAVE_PATH_IMG)

print("Saved complete file to :", SAVE_PATH_IMG)

In [ ]:
img_data       = np.load(SAVE_PATH_IMG, allow_pickle=True)
img_embeddings = img_data['embeddings']
img_names      = img_data['image_names']

print("Final shape    :", img_embeddings.shape)
print("Unique images  :", len(set(img_names)))
print("Expected       : (8091, 256)")

In [ ]:
# ── Figure 4.3: BLIP Image Embedding Extraction Output ───────────
import numpy as np
import os

# ⬇ Update this path to where you saved your embeddings on Drive
EMB_PATH = '/content/drive/MyDrive/flickr8k/image_embeddings.npz'

# Load the saved embeddings
data = np.load(EMB_PATH)

# Try common key names (use whichever your notebook saved)
possible_keys = ['embeddings', 'image_embeddings', 'arr_0', 'features']
emb = None
for key in possible_keys:
    if key in data:
        emb = data[key]
        print(f"  Key used      : '{key}'")
        break

if emb is None:
    print("Keys available:", list(data.keys()))
else:
    print("=" * 50)
    print("  BLIP Image Embedding Extraction — Summary")
    print("=" * 50)
    print(f"  File           : {os.path.basename(EMB_PATH)}")
    print(f"  Final shape    : {emb.shape}")
    print(f"  Unique images  : {emb.shape[0]}")
    print(f"  Embedding dim  : {emb.shape[1]}")
    print(f"  Data type      : {emb.dtype}")
    print(f"  File size      : {os.path.getsize(EMB_PATH) / (1024*1024):.2f} MB")
    print(f"  Min value      : {emb.min():.6f}")
    print(f"  Max value      : {emb.max():.6f}")
    print(f"  L2 norm sample : {np.linalg.norm(emb[0]):.6f}  (should be ~1.0)")
    print("=" * 50)
    print("   Embeddings verified and ready for cosine scoring")

**Caption Embeddings**

In [ ]:
BATCH_SIZE       = 64
CHECKPOINT_EVERY = 500

if os.path.exists(SAVE_PATH_TXT):
    try:
        checkpoint    = np.load(SAVE_PATH_TXT, allow_pickle=True)
        cap_emb_list  = list(checkpoint['embeddings'])
        cap_name_list = list(checkpoint['image_names'])
        cap_text_list = list(checkpoint['captions'])
        already_done  = len(cap_emb_list)
        print(f"Resuming from checkpoint : {already_done} captions already done")
    except Exception as e:
        print(f"Checkpoint corrupted, starting fresh : {e}")
        os.remove(SAVE_PATH_TXT)
        cap_emb_list  = []
        cap_name_list = []
        cap_text_list = []
        already_done  = 0
else:
    cap_emb_list  = []
    cap_name_list = []
    cap_text_list = []
    already_done  = 0
    print("No checkpoint found, starting fresh")

remaining_df = df.iloc[already_done:].reset_index(drop=True)
total        = len(df)
batches      = [remaining_df.iloc[i:i+BATCH_SIZE] for i in range(0, len(remaining_df), BATCH_SIZE)]

print(f"Total captions  : {total}")
print(f"Remaining       : {len(remaining_df)}")
print(f"Batch size      : {BATCH_SIZE}")
print(f"Batches left    : {len(batches)}")

SAVE_BASE_TXT      = SAVE_PATH_TXT.replace(".npz", "")
SAVE_BASE_TXT_TEMP = SAVE_BASE_TXT + "_temp"

for batch_idx, batch in enumerate(batches):

    captions  = batch['caption_clean'].tolist()
    img_names = batch['image_name'].tolist()

    try:
        inputs = processor(
            text           = captions,
            return_tensors = "pt",
            padding        = True,
            truncation     = True,
            max_length     = 512
        ).to(DEVICE)

        with torch.no_grad():
            text_out = itm_model.text_encoder(
                input_ids      = inputs["input_ids"],
                attention_mask = inputs["attention_mask"]
            )
            # text_proj maps into shared 256-dim space with image
            text_feat  = itm_model.text_proj(
                text_out.last_hidden_state[:, 0, :]
            )
            embeddings = text_feat.cpu().numpy()
            embeddings = normalize(embeddings, norm="l2")

        cap_emb_list.extend(embeddings)
        cap_name_list.extend(img_names)
        cap_text_list.extend(captions)

    except Exception as e:
        print(f"Batch {batch_idx} failed: {e}")
        continue

    processed_so_far = already_done + (batch_idx + 1) * BATCH_SIZE

    if processed_so_far % CHECKPOINT_EVERY < BATCH_SIZE:
        np.savez_compressed(
            SAVE_BASE_TXT_TEMP,
            image_names = np.array(cap_name_list),
            captions    = np.array(cap_text_list),
            embeddings  = np.array(cap_emb_list)
        )
        if os.path.exists(SAVE_PATH_TXT):
            os.remove(SAVE_PATH_TXT)
        os.rename(SAVE_BASE_TXT_TEMP + ".npz", SAVE_PATH_TXT)
        print(f"Checkpoint saved at ~{len(cap_emb_list)}/{total}")

np.savez_compressed(
    SAVE_BASE_TXT_TEMP,
    image_names = np.array(cap_name_list),
    captions    = np.array(cap_text_list),
    embeddings  = np.array(cap_emb_list)
)
if os.path.exists(SAVE_PATH_TXT):
    os.remove(SAVE_PATH_TXT)
os.rename(SAVE_BASE_TXT_TEMP + ".npz", SAVE_PATH_TXT)

print(f"\nDone. Total saved   : {len(cap_emb_list)}")
print(f"Embedding shape     : {np.array(cap_emb_list).shape}")
print(f"Saved to            : {SAVE_PATH_TXT}")
print(f"Expected shape      : (40455, 256)")

 Load Both Embeddings and Show a Sample

In [ ]:
IMAGES_PATH = "/content/drive/MyDrive/flickr8k/Images"


# Load image and caption embeddings
img_data       = np.load(SAVE_PATH_IMG, allow_pickle=True)
img_embeddings = img_data['embeddings']
img_names      = img_data['image_names']

txt_data       = np.load(SAVE_PATH_TXT, allow_pickle=True)
cap_embeddings = txt_data['embeddings']
cap_names      = txt_data['image_names']
cap_texts      = txt_data['captions']

sample_img_name = img_names[0]
sample_img_vec  = img_embeddings[0].reshape(1, -1)
matched_idx     = np.where(cap_names == sample_img_name)[0]

scores = []
for idx in matched_idx:
    score = cosine_similarity(sample_img_vec, cap_embeddings[idx].reshape(1, -1))[0][0]
    scores.append((cap_texts[idx], score))

scores = sorted(scores, key=lambda x: x[1], reverse=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left side — show image
img_path = os.path.join(IMAGES_PATH, sample_img_name)
image    = Image.open(img_path).convert("RGB")
axes[0].imshow(image)
axes[0].axis("off")
axes[0].set_title(sample_img_name, fontsize=9)

# Right side — show captions and scores as horizontal bars
captions_short = [c[:55] + "..." if len(c) > 55 else c for c, s in scores]
score_values   = [s for c, s in scores]
colors         = ["#2ecc71" if s == max(score_values) else "#3498db" for s in score_values]

bars = axes[1].barh(captions_short, score_values, color=colors)
axes[1].set_xlim(0, 1)
axes[1].set_xlabel("Cosine Similarity Score")
axes[1].set_title("Captions Ranked by Cosine Similarity")
axes[1].invert_yaxis()

for bar, score in zip(bars, score_values):
    axes[1].text(
        bar.get_width() + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{score:.4f}",
        va="center",
        fontsize=10
    )

plt.tight_layout()
plt.savefig("sample_ranking.png", dpi=150, bbox_inches="tight")
plt.show()

print("Best caption  :", scores[0][0])
print("Best score    :", round(scores[0][1], 4))
print("Worst caption :", scores[-1][0])
print("Worst score   :", round(scores[-1][1], 4))

### Full Cosine Similarity Computation

In [ ]:
results = []
total   = len(img_names)

print(f"Processing {total} images...")

for i, img_name in enumerate(img_names):

    img_vec     = img_embeddings[i].reshape(1, -1)
    matched_idx = np.where(cap_names == img_name)[0]

    if len(matched_idx) == 0:
        continue

    matched_caps   = cap_embeddings[matched_idx]
    matched_texts  = cap_texts[matched_idx]
    scores         = cosine_similarity(img_vec, matched_caps)[0]

    ranked_idx     = np.argsort(scores)[::-1]
    ranked_caps    = matched_texts[ranked_idx]
    ranked_scores  = scores[ranked_idx]

    results.append({
        "image_name"   : img_name,
        "best_caption" : ranked_caps[0],
        "best_score"   : round(float(ranked_scores[0]), 4),
        "worst_caption": ranked_caps[-1],
        "worst_score"  : round(float(ranked_scores[-1]), 4),
        "mean_score"   : round(float(scores.mean()), 4),
        "score_spread" : round(float(ranked_scores[0] - ranked_scores[-1]), 4),
        "all_captions" : list(ranked_caps),
        "all_scores"   : [round(float(s), 4) for s in ranked_scores]
    })

    if (i + 1) % 1000 == 0:
        print(f"Progress : {i+1}/{total}")

results_df = pd.DataFrame(results)

print(f"\nDone. Total images processed : {len(results_df)}")
print(f"\nSample output:")
print(results_df[['image_name','best_score','worst_score','mean_score','score_spread']].head(5))

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Figure 3.7 — Cosine Similarity Score Distribution Plot
# ─────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast

# ── 1. Load cosine scores CSV ───────────────────────────────
# Change this path to wherever your file is saved on Drive
CSV_PATH = '/content/drive/MyDrive/flickr8k/ranking scores/cosine_scores.csv'   # <-- update if needed

df = pd.read_csv(CSV_PATH)
print("Columns found:", df.columns.tolist())
print("Shape:", df.shape)
print(df.head(2))

In [ ]:
CSV_PATH  = '/content/drive/MyDrive/flickr8k/ranking scores/cosine_scores.csv'
SAVE_PATH = '/content/drive/MyDrive/flickr8k/figure_5_5_cosine_distribution.png'

df = pd.read_csv(CSV_PATH)

# Parse the list string into actual list
df['scores_list'] = df['cosine_scores'].apply(ast.literal_eval)

best_scores  = np.array(df['scores_list'].apply(max))
worst_scores = np.array(df['scores_list'].apply(min))
avg_scores   = np.array(df['scores_list'].apply(np.mean))

print(f"Best  — Mean: {best_scores.mean():.4f}")
print(f"Worst — Mean: {worst_scores.mean():.4f}")
print(f"Avg   — Mean: {avg_scores.mean():.4f}")

# ── Plot ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle(
    'Cosine Similarity Score Distributions — Flickr8k Dataset (8,091 Images)',
    fontsize=13, fontweight='bold', y=1.02
)

configs = [
    (best_scores,  '#1E8449', 'Best Caption Score (per image)',
     f'Mean = {best_scores.mean():.4f}'),
    (worst_scores, '#C0392B', 'Worst Caption Score (per image)',
     f'Mean = {worst_scores.mean():.4f}'),
    (avg_scores,   '#1A5276', 'Average Caption Score (per image)',
     f'Mean = {avg_scores.mean():.4f}'),
]

for ax, (scores, color, title, mean_label) in zip(axes, configs):
    ax.hist(scores, bins=50, color=color, edgecolor='white',
            linewidth=0.4, alpha=0.85)
    ax.axvline(scores.mean(), color='black', linestyle='--',
               linewidth=1.8, label=mean_label)
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
    ax.set_xlabel('Cosine Similarity', fontsize=10)
    ax.set_ylabel('Number of Images', fontsize=10)
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(SAVE_PATH, dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved → {SAVE_PATH}")

In [ ]:
# Save the result to CSV

RESULTS_SAVE_PATH = "/content/drive/MyDrive/flickr8k/refinement_results.csv"

results_df.to_csv(RESULTS_SAVE_PATH, index=False)

print("Saved to          :", RESULTS_SAVE_PATH)
print("Total rows        :", len(results_df))
print("\nOverall Statistics:")
print(f"  Mean best score   : {results_df['best_score'].mean():.4f}")
print(f"  Mean worst score  : {results_df['worst_score'].mean():.4f}")
print(f"  Mean avg score    : {results_df['mean_score'].mean():.4f}")
print(f"  Mean spread       : {results_df['score_spread'].mean():.4f}")
print(f"  Best score ever   : {results_df['best_score'].max():.4f}")
print(f"  Worst score ever  : {results_df['worst_score'].min():.4f}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

# ── STEP 1: Inspect npz files first ─────────────────────────
IMG_EMB_PATH = '/content/drive/MyDrive/flickr8k/image_embeddings.npz'
CAP_EMB_PATH = '/content/drive/MyDrive/flickr8k/caption_embeddings.npz'

img_data = np.load(IMG_EMB_PATH, allow_pickle=True)
cap_data = np.load(CAP_EMB_PATH, allow_pickle=True)

print("Image npz keys and shapes:")
for k in img_data.files:
    print(f"  '{k}' → {img_data[k].shape}  dtype={img_data[k].dtype}")

print("\nCaption npz keys and shapes:")
for k in cap_data.files:
    print(f"  '{k}' → {cap_data[k].shape}  dtype={cap_data[k].dtype}")

### Distribution Plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(results_df['best_score'],  bins=40, color="#2ecc71", edgecolor="black")
axes[0].axvline(results_df['best_score'].mean(), color="red", linestyle="--")
axes[0].set_title("Best Caption Score Distribution")
axes[0].set_xlabel("Cosine Similarity")
axes[0].set_ylabel("Number of Images")

axes[1].hist(results_df['mean_score'],  bins=40, color="#3498db", edgecolor="black")
axes[1].axvline(results_df['mean_score'].mean(), color="red", linestyle="--")
axes[1].set_title("Mean Caption Score Distribution")
axes[1].set_xlabel("Cosine Similarity")
axes[1].set_ylabel("Number of Images")

axes[2].hist(results_df['score_spread'], bins=40, color="#9b59b6", edgecolor="black")
axes[2].axvline(results_df['score_spread'].mean(), color="red", linestyle="--")
axes[2].set_title("Score Spread Distribution")
axes[2].set_xlabel("Best Score - Worst Score")
axes[2].set_ylabel("Number of Images")

plt.suptitle("BLIP ITM Cosine Similarity Analysis — Flickr8k", fontsize=13)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/flickr8k/cosine_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print("Plot saved to Drive")

**Interpretation of the Three Plots**

***Best Caption Score Distribution (green)*** shows that when BLIP picks the single best matching caption for each image, scores cluster around 0.52 to 0.55. This means the model consistently finds at least one caption that aligns reasonably well with every image — no image was left without a decent match.

***Mean Caption Score Distribution (blue)*** shows the average score across all 5 captions per image sits around 0.48 to 0.50. This is slightly lower than the best score, which is expected — some of the 5 captions describe the image loosely while others describe it precisely. The gap between green and blue plots confirms the refinement is working — picking the best caption gives you a meaningfully better result than picking randomly.

***Score Spread Distribution (purple)*** shows that for most images the difference between the best and worst caption is small — around 0.08 to 0.10. This tells you two things. First, all 5 Flickr8k captions for any given image are semantically similar to each other, which makes sense since human annotators described the same image. Second, cosine similarity is still able to distinguish between them and rank them, which validates the refinement approach.
Overall conclusion — the embeddings are well aligned, the refinement is working, and the model is consistently identifying the most descriptive caption for each image across the full 8091 image dataset.

In [ ]:
# Show best and worst ranked image

top5    = results_df.nlargest(5, 'best_score')
bottom5 = results_df.nsmallest(5, 'best_score')

fig, axes = plt.subplots(2, 5, figsize=(20, 9))

for i, (_, row) in enumerate(top5.iterrows()):
    img  = Image.open(os.path.join(IMAGES_PATH, row['image_name'])).convert("RGB")
    axes[0][i].imshow(img)
    axes[0][i].axis("off")
    axes[0][i].set_title(
        f"Score: {row['best_score']}\n{row['best_caption'][:40]}...",
        fontsize=7
    )

for i, (_, row) in enumerate(bottom5.iterrows()):
    img  = Image.open(os.path.join(IMAGES_PATH, row['image_name'])).convert("RGB")
    axes[1][i].imshow(img)
    axes[1][i].axis("off")
    axes[1][i].set_title(
        f"Score: {row['best_score']}\n{row['best_caption'][:40]}...",
        fontsize=7
    )

axes[0][0].set_ylabel("Top 5", fontsize=12)
axes[1][0].set_ylabel("Bottom 5", fontsize=12)

plt.suptitle("Top 5 vs Bottom 5 Image-Caption Alignment", fontsize=13)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/flickr8k/top_bottom_ranking.png", dpi=150, bbox_inches="tight")
plt.show()

Full Cosine Similarity Computation

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

results = []
total   = len(img_names)

print(f"Processing {total} images...")

for i, img_name in enumerate(img_names):

    img_vec     = img_embeddings[i].reshape(1, -1)
    matched_idx = np.where(cap_names == img_name)[0]

    if len(matched_idx) == 0:
        continue

    matched_caps   = cap_embeddings[matched_idx]
    matched_texts  = cap_texts[matched_idx]
    scores         = cosine_similarity(img_vec, matched_caps)[0]

    ranked_idx     = np.argsort(scores)[::-1]
    ranked_caps    = matched_texts[ranked_idx]
    ranked_scores  = scores[ranked_idx]

    results.append({
        "image_name"   : img_name,
        "best_caption" : ranked_caps[0],
        "best_score"   : round(float(ranked_scores[0]), 4),
        "worst_caption": ranked_caps[-1],
        "worst_score"  : round(float(ranked_scores[-1]), 4),
        "mean_score"   : round(float(scores.mean()), 4),
        "score_spread" : round(float(ranked_scores[0] - ranked_scores[-1]), 4),
        "all_captions" : list(ranked_caps),
        "all_scores"   : [round(float(s), 4) for s in ranked_scores]
    })

    if (i + 1) % 1000 == 0:
        print(f"Progress : {i+1}/{total}")

results_df = pd.DataFrame(results)

print(f"\nDone. Total images processed : {len(results_df)}")
print(f"\nSample output:")
print(results_df[['image_name','best_score','worst_score','mean_score','score_spread']].head(5))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np # Add numpy import for np.where and np.argsort

# Code to define results_df (copied from guLouNMFbHMi)
results = []
total   = len(img_names)

print(f"Processing {total} images...")

for i, img_name in enumerate(img_names):

    img_vec     = img_embeddings[i].reshape(1, -1)
    matched_idx = np.where(cap_names == img_name)[0]

    if len(matched_idx) == 0:
        continue

    matched_caps   = cap_embeddings[matched_idx]
    matched_texts  = cap_texts[matched_idx]
    scores         = cosine_similarity(img_vec, matched_caps)[0]

    ranked_idx     = np.argsort(scores)[::-1]
    ranked_caps    = matched_texts[ranked_idx]
    ranked_scores  = scores[ranked_idx]

    results.append({
        "image_name"   : img_name,
        "best_caption" : ranked_caps[0],
        "best_score"   : round(float(ranked_scores[0]), 4),
        "worst_caption": ranked_caps[-1],
        "worst_score"  : round(float(ranked_scores[-1]), 4),
        "mean_score"   : round(float(scores.mean()), 4),
        "score_spread" : round(float(ranked_scores[0] - ranked_scores[-1]), 4),
        "all_captions" : list(ranked_caps),
        "all_scores"   : [round(float(s), 4) for s in ranked_scores]
    })

    if (i + 1) % 1000 == 0:
        print(f"Progress : {i+1}/{total}")

results_df = pd.DataFrame(results)

print(f"\nDone. Total images processed : {len(results_df)}")
print(f"\nSample output:")
print(results_df[['image_name','best_score','worst_score','mean_score','score_spread']].head(5))

# Original content of cell qaXifaC0bHRU
RESULTS_SAVE_PATH = "/content/drive/MyDrive/flickr8k/refinement_results.csv"

results_df.to_csv(RESULTS_SAVE_PATH, index=False)

print("Saved to          :", RESULTS_SAVE_PATH)
print("Total rows        :", len(results_df))
print("\nOverall Statistics:")
print(f"  Mean best score   : {results_df['best_score'].mean():.4f}")
print(f"  Mean worst score  : {results_df['worst_score'].mean():.4f}")
print(f"  Mean avg score    : {results_df['mean_score'].mean():.4f}")
print(f"  Mean spread       : {results_df['score_spread'].mean():.4f}")
print(f"  Best score ever   : {results_df['best_score'].max():.4f}")
print(f"  Worst score ever  : {results_df['worst_score'].min():.4f}")

Distribution Plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(results_df['best_score'],  bins=40, color="#2ecc71", edgecolor="black")
axes[0].axvline(results_df['best_score'].mean(), color="red", linestyle="--")
axes[0].set_title("Best Caption Score Distribution")
axes[0].set_xlabel("Cosine Similarity")
axes[0].set_ylabel("Number of Images")

axes[1].hist(results_df['mean_score'],  bins=40, color="#3498db", edgecolor="black")
axes[1].axvline(results_df['mean_score'].mean(), color="red", linestyle="--")
axes[1].set_title("Mean Caption Score Distribution")
axes[1].set_xlabel("Cosine Similarity")
axes[1].set_ylabel("Number of Images")

axes[2].hist(results_df['score_spread'], bins=40, color="#9b59b6", edgecolor="black")
axes[2].axvline(results_df['score_spread'].mean(), color="red", linestyle="--")
axes[2].set_title("Score Spread Distribution")
axes[2].set_xlabel("Best Score - Worst Score")
axes[2].set_ylabel("Number of Images")

plt.suptitle("BLIP ITM Cosine Similarity Analysis — Flickr8k", fontsize=13)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/flickr8k/cosine_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print("Plot saved to Drive")

**Interpretation of the Three Plots**

***Best Caption Score Distribution (green)*** shows that when BLIP picks the single best matching caption for each image, scores cluster around 0.52 to 0.55. This means the model consistently finds at least one caption that aligns reasonably well with every image — no image was left without a decent match.

***Mean Caption Score Distribution (blue)*** shows the average score across all 5 captions per image sits around 0.48 to 0.50. This is slightly lower than the best score, which is expected — some of the 5 captions describe the image loosely while others describe it precisely. The gap between green and blue plots confirms the refinement is working — picking the best caption gives you a meaningfully better result than picking randomly.

***Score Spread Distribution (purple)*** shows that for most images the difference between the best and worst caption is small — around 0.08 to 0.10. This tells you two things. First, all 5 Flickr8k captions for any given image are semantically similar to each other, which makes sense since human annotators described the same image. Second, cosine similarity is still able to distinguish between them and rank them, which validates the refinement approach.
Overall conclusion — the embeddings are well aligned, the refinement is working, and the model is consistently identifying the most descriptive caption for each image across the full 8091 image dataset.

In [ ]:
# Show best and worst ranked image

top5    = results_df.nlargest(5, 'best_score')
bottom5 = results_df.nsmallest(5, 'best_score')

fig, axes = plt.subplots(2, 5, figsize=(20, 9))

for i, (_, row) in enumerate(top5.iterrows()):
    img  = Image.open(os.path.join(IMAGES_PATH, row['image_name'])).convert("RGB")
    axes[0][i].imshow(img)
    axes[0][i].axis("off")
    axes[0][i].set_title(
        f"Score: {row['best_score']}\n{row['best_caption'][:40]}...",
        fontsize=7
    )

for i, (_, row) in enumerate(bottom5.iterrows()):
    img  = Image.open(os.path.join(IMAGES_PATH, row['image_name'])).convert("RGB")
    axes[1][i].imshow(img)
    axes[1][i].axis("off")
    axes[1][i].set_title(
        f"Score: {row['best_score']}\n{row['best_caption'][:40]}...",
        fontsize=7
    )

axes[0][0].set_ylabel("Top 5", fontsize=12)
axes[1][0].set_ylabel("Bottom 5", fontsize=12)

plt.suptitle("Top 5 vs Bottom 5 Image-Caption Alignment", fontsize=13)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/flickr8k/top_bottom_ranking.png", dpi=150, bbox_inches="tight")
plt.show()

**Model Ensembling**

In [ ]:
def safe_save(df, path):
    """Save CSV with atomic write to prevent corruption."""
    tmp = path + ".tmp"
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)
    print(f"Saved: {path}  ({len(df)} rows, {os.path.getsize(path)/1024:.1f} KB)")

In [ ]:
import os
import ast
import torch
import numpy as np
import pandas as pd
from PIL import Image
from transformers import BlipProcessor, BlipForImageTextRetrieval
from sklearn.metrics.pairwise import cosine_similarity
import gc

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMAGES_PATH   = "/content/drive/MyDrive/flickr8k/Images"
CAPTIONS_PATH = "/content/drive/MyDrive/flickr8k/Flickr_TextData/Flickr8k.token.txt"
SAVE_PATH_IMG = "/content/drive/MyDrive/flickr8k/image_embeddings.npz"
SAVE_PATH_TXT = "/content/drive/MyDrive/flickr8k/caption_embeddings.npz"
ITM_SAVE_PATH = "/content/drive/MyDrive/flickr8k/ranking scores/itm_scores.csv"

# Load embeddings
img_data       = np.load(SAVE_PATH_IMG, allow_pickle=True)
img_embeddings = img_data['embeddings']
img_names      = img_data['image_names']

txt_data       = np.load(SAVE_PATH_TXT, allow_pickle=True)
cap_embeddings = txt_data['embeddings']
cap_names      = txt_data['image_names']
cap_texts      = txt_data['captions']

# Load captions
captions_data = []
with open(CAPTIONS_PATH, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split('\t')
        if len(parts) == 2:
            img_caption_id, caption = parts
            img_name = img_caption_id.split('#')[0]
            captions_data.append({
                'image_name': img_name,
                'caption'   : caption
            })

df = pd.DataFrame(captions_data)

print("Device             :", DEVICE)
print("Image embeddings   :", img_embeddings.shape)
print("Caption embeddings :", cap_embeddings.shape)
print("Total captions     :", len(df))
print("Unique images      :", df['image_name'].nunique())
print("VRAM allocated     :", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

In [ ]:
ITM_MODEL_NAME     = "Salesforce/blip-itm-large-coco"
CAPTION_MODEL_NAME = "Salesforce/blip-image-captioning-large"

gc.collect()
torch.cuda.empty_cache()

print("Loading processor...")
processor = BlipProcessor.from_pretrained(CAPTION_MODEL_NAME)
print("Processor loaded")

print("Loading ITM model...")
itm_model = BlipForImageTextRetrieval.from_pretrained(
    ITM_MODEL_NAME,
    torch_dtype = torch.float16
).to(DEVICE)
itm_model.eval()
print("ITM model on :", next(itm_model.parameters()).device)
print("VRAM after loading :", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

In [ ]:
CHECKPOINT_EVERY = 100

if os.path.exists(ITM_SAVE_PATH):
    try:
        done_df      = pd.read_csv(ITM_SAVE_PATH)
        already_done = set(done_df['image_name'].tolist())
        all_results  = done_df.to_dict('records')
        print(f"Resuming from checkpoint : {len(already_done)} images done")
    except Exception as e:
        print(f"Checkpoint corrupted, starting fresh : {e}")
        os.remove(ITM_SAVE_PATH)
        already_done = set()
        all_results  = []
else:
    already_done = set()
    all_results  = []
    print("No checkpoint found, starting fresh")

remaining = [img for img in img_names if img not in already_done]
total     = len(img_names)

print(f"Total images    : {total}")
print(f"Remaining       : {len(remaining)}")
print(f"Starting ITM scoring...")

for i, img_name in enumerate(remaining):

    img_path    = os.path.join(IMAGES_PATH, img_name)
    cap_indices = np.where(cap_names == img_name)[0]
    captions    = list(cap_texts[cap_indices])

    if not os.path.exists(img_path):
        print(f"Image not found, skipping : {img_name}")
        continue

    if len(captions) == 0:
        print(f"No captions, skipping : {img_name}")
        continue

    try:
        image  = Image.open(img_path).convert("RGB")
        scores = []

        for caption in captions:
            inputs = processor(
                images         = image,
                text           = caption,
                return_tensors = "pt",
                padding        = True
            ).to(DEVICE)

            with torch.no_grad():
                output    = itm_model(**inputs)
                itm_score = torch.nn.functional.softmax(
                    output.itm_score, dim=1
                )[:, 1].item()

            scores.append(round(itm_score, 4))

        # Select top 2 by ITM score
        ranked_idx    = np.argsort(scores)[::-1]
        top2_idx      = ranked_idx[:2]
        top2_caption1 = captions[top2_idx[0]]
        top2_caption2 = captions[top2_idx[1]]
        top2_score1   = scores[top2_idx[0]]
        top2_score2   = scores[top2_idx[1]]

        all_results.append({
            "image_name"   : img_name,
            "captions"     : str(captions),
            "itm_scores"   : str(scores),
            "itm_top2_cap1": top2_caption1,
            "itm_top2_cap2": top2_caption2,
            "itm_top2_idx1": int(top2_idx[0]),
            "itm_top2_idx2": int(top2_idx[1]),
            "itm_score1"   : top2_score1,
            "itm_score2"   : top2_score2
        })

    except Exception as e:
        print(f"Failed {img_name} : {e}")
        continue

    if (i + 1) % 50 == 0:
        print(f"Progress : {len(all_results)}/{total}")

    if (i + 1) % CHECKPOINT_EVERY == 0:
        temp_path = ITM_SAVE_PATH + ".tmp"
        pd.DataFrame(all_results).to_csv(temp_path, index=False)
        os.replace(temp_path, ITM_SAVE_PATH)
        print(f"Checkpoint saved : {len(all_results)}/{total}")

# Final save
temp_path = ITM_SAVE_PATH + ".tmp"
pd.DataFrame(all_results).to_csv(temp_path, index=False)
os.replace(temp_path, ITM_SAVE_PATH)

print(f"\nDone")
print(f"Total saved : {len(all_results)}")
print(f"Saved to    : {ITM_SAVE_PATH}")

In [ ]:
itm_df = pd.read_csv(ITM_SAVE_PATH)

print("Total rows        :", len(itm_df))
print("Expected          : 8091")
print("Columns           :", itm_df.columns.tolist())

print("\nSample row:")
row = itm_df.iloc[0]
print(f"  Image      : {row['image_name']}")
print(f"  ITM Top 1  : {row['itm_score1']:.4f} | {row['itm_top2_cap1']}")
print(f"  ITM Top 2  : {row['itm_score2']:.4f} | {row['itm_top2_cap2']}")

In [ ]:
'''import subprocess
subprocess.run(["pip", "install", "einops", "timm", "-q"])'''

from transformers import AutoModel

gc.collect()
torch.cuda.empty_cache()

print("Loading Jina M0...")
jina_model = AutoModel.from_pretrained(
    "jinaai/jina-reranker-m0",
    trust_remote_code = True,
    torch_dtype       = torch.float16
).to(DEVICE)
jina_model.eval()

print("Jina M0 on   :", next(jina_model.parameters()).device)
print("VRAM after   :", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

In [ ]:
# jina reranker with top2
JINA_SAVE_PATH   = "/content/drive/MyDrive/flickr8k/ranking scores/jina_scores.csv"
CHECKPOINT_EVERY = 100

if os.path.exists(JINA_SAVE_PATH):
    try:
        done_df      = pd.read_csv(JINA_SAVE_PATH)
        already_done = set(done_df['image_name'].tolist())
        all_results  = done_df.to_dict('records')
        print(f"Resuming from checkpoint : {len(already_done)} images done")
    except Exception as e:
        print(f"Checkpoint corrupted, starting fresh : {e}")
        os.remove(JINA_SAVE_PATH)
        already_done = set()
        all_results  = []
else:
    already_done = set()
    all_results  = []
    print("No checkpoint found, starting fresh")

remaining = [img for img in img_names if img not in already_done]
total     = len(img_names)

print(f"Total images    : {total}")
print(f"Remaining       : {len(remaining)}")
print(f"Starting Jina scoring...")

for i, img_name in enumerate(remaining):

    img_path    = os.path.join(IMAGES_PATH, img_name)
    cap_indices = np.where(cap_names == img_name)[0]
    captions    = list(cap_texts[cap_indices])

    if not os.path.exists(img_path):
        print(f"Image not found, skipping : {img_name}")
        continue

    if len(captions) == 0:
        print(f"No captions, skipping : {img_name}")
        continue

    try:
        image  = Image.open(img_path).convert("RGB")
        pairs  = [[caption, image] for caption in captions]

        with torch.no_grad():
            scores = jina_model.compute_score(
                pairs,
                query_type = "text",
                doc_type   = "image"
            )

        scores = [round(float(s), 4) for s in scores]

        # Select top 2 by Jina score
        ranked_idx    = np.argsort(scores)[::-1]
        top2_idx      = ranked_idx[:2]
        top2_caption1 = captions[top2_idx[0]]
        top2_caption2 = captions[top2_idx[1]]
        top2_score1   = scores[top2_idx[0]]
        top2_score2   = scores[top2_idx[1]]

        all_results.append({
            "image_name"     : img_name,
            "captions"       : str(captions),
            "jina_scores"    : str(scores),
            "jina_top2_cap1" : top2_caption1,
            "jina_top2_cap2" : top2_caption2,
            "jina_top2_idx1" : int(top2_idx[0]),
            "jina_top2_idx2" : int(top2_idx[1]),
            "jina_score1"    : top2_score1,
            "jina_score2"    : top2_score2
        })

    except Exception as e:
        print(f"Failed {img_name} : {e}")
        continue

    if (i + 1) % 50 == 0:
        print(f"Progress : {len(all_results)}/{total}")

    if (i + 1) % CHECKPOINT_EVERY == 0:
        temp_path = JINA_SAVE_PATH + ".tmp"
        pd.DataFrame(all_results).to_csv(temp_path, index=False)
        os.replace(temp_path, JINA_SAVE_PATH)
        print(f"Checkpoint saved : {len(all_results)}/{total}")

# Final save
temp_path = JINA_SAVE_PATH + ".tmp"
pd.DataFrame(all_results).to_csv(temp_path, index=False)
os.replace(temp_path, JINA_SAVE_PATH)

print(f"\nDone")
print(f"Total saved : {len(all_results)}")
print(f"Saved to    : {JINA_SAVE_PATH}")

In [ ]:
jina_df = pd.read_csv(JINA_SAVE_PATH)

print("Total rows        :", len(jina_df))
print("Expected          : 8091")
print("Columns           :", jina_df.columns.tolist())

row = jina_df.iloc[0]
print(f"\nSample row:")
print(f"  Image      : {row['image_name']}")
print(f"  Jina Top 1 : {row['jina_score1']:.4f} | {row['jina_top2_cap1']}")
print(f"  Jina Top 2 : {row['jina_score2']:.4f} | {row['jina_top2_cap2']}")

In [ ]:
COSINE_SAVE_PATH = "/content/drive/MyDrive/flickr8k/ranking scores/cosine_scores.csv"

already_done = set()
all_results  = []

total = len(img_names)
print(f"Total images    : {total}")
print(f"Starting Cosine scoring...")

for i, img_name in enumerate(img_names):

    cap_indices = np.where(cap_names == img_name)[0]
    captions    = list(cap_texts[cap_indices])

    if len(captions) == 0:
        print(f"No captions, skipping : {img_name}")
        continue

    img_vec  = img_embeddings[i].reshape(1, -1)
    cap_vecs = cap_embeddings[cap_indices]
    scores   = cosine_similarity(img_vec, cap_vecs)[0]
    scores   = [round(float(s), 4) for s in scores]

    ranked_idx    = np.argsort(scores)[::-1]
    top2_idx      = ranked_idx[:2]
    top2_caption1 = captions[top2_idx[0]]
    top2_caption2 = captions[top2_idx[1]]
    top2_score1   = scores[top2_idx[0]]
    top2_score2   = scores[top2_idx[1]]

    all_results.append({
        "image_name"       : img_name,
        "captions"         : str(captions),
        "cosine_scores"    : str(scores),
        "cosine_top2_cap1" : top2_caption1,
        "cosine_top2_cap2" : top2_caption2,
        "cosine_top2_idx1" : int(top2_idx[0]),
        "cosine_top2_idx2" : int(top2_idx[1]),
        "cosine_score1"    : top2_score1,
        "cosine_score2"    : top2_score2
    })

    if (i + 1) % 1000 == 0:
        print(f"Progress : {i+1}/{total}")

cosine_df = pd.DataFrame(all_results)
cosine_df.to_csv(COSINE_SAVE_PATH, index=False)

print(f"\nDone")
print(f"Total saved : {len(cosine_df)}")
print(f"Saved to    : {COSINE_SAVE_PATH}")

In [ ]:
cosine_df = pd.read_csv(COSINE_SAVE_PATH)

print("Total rows        :", len(cosine_df))
print("Expected          : 8091")
print("Columns           :", cosine_df.columns.tolist())

row = cosine_df.iloc[0]
print(f"\nSample row:")
print(f"  Image         : {row['image_name']}")
print(f"  Cosine Top 1  : {row['cosine_score1']:.4f} | {row['cosine_top2_cap1']}")
print(f"  Cosine Top 2  : {row['cosine_score2']:.4f} | {row['cosine_top2_cap2']}")

**Majority Voting**

In [ ]:
import pandas as pd

import re
from collections import Counter

# ── Load from INPUT (permanent) ──
itm_df    = pd.read_csv("/content/drive/MyDrive/flickr8k/ranking scores/itm_scores.csv")
jina_df   = pd.read_csv("/content/drive/MyDrive/flickr8k/ranking scores/jina_scores.csv")
cosine_df = pd.read_csv("/content/drive/MyDrive/flickr8k/ranking scores/cosine_scores.csv")


print("ITM rows    :", len(itm_df))
print("Jina rows   :", len(jina_df))
print("Cosine rows :", len(cosine_df))

# ── Merge all 3 on image_name ──
df = itm_df[['image_name','captions','itm_top2_idx1','itm_top2_idx2']].merge(
     jina_df[['image_name','jina_top2_idx1','jina_top2_idx2']], on='image_name').merge(
     cosine_df[['image_name','cosine_top2_idx1','cosine_top2_idx2']], on='image_name')

print("Merged rows :", len(df))

# ── Safe caption parser ──
def parse_captions(val):
    """Safely parse captions column regardless of format."""
    try:
        return ast.literal_eval(val)
    except:
        pass
    try:
        # Extract all quoted strings from the list-like string
        matches = re.findall(r"'(.*?)'(?:\s*,|\s*\])", str(val))
        if len(matches) >= 2:
            return matches
        # fallback: split by comma inside brackets
        val = str(val).strip().strip('[]')
        return [v.strip().strip("'\"") for v in val.split("',")]
    except:
        return []

# ── Quick check on 1 row ──
sample_caps = parse_captions(df['captions'].iloc[0])
print(f"\nParser check — {len(sample_caps)} captions found:")
for c in sample_caps:
    print(f"  → {c}")

In [ ]:
import re

def parse_captions(val):
    """Extract captions stored in np.str_('...') format."""
    matches = re.findall(r"np\.str_\('(.*?)'\)", str(val))
    if matches:
        return matches
    # fallback for normal list format
    try:
        return ast.literal_eval(val)
    except:
        return []

# ── Verify ──
sample_caps = parse_captions(df['captions'].iloc[0])
print(f"Captions found: {len(sample_caps)}")
for c in sample_caps:
    print(f"  → {c}")

In [ ]:
# ── Majority Voting ──
voted_results = []
skipped = 0

for _, row in df.iterrows():
    captions = parse_captions(row['captions'])

    if len(captions) < 2:
        skipped += 1
        continue

    votes = [
        int(row['itm_top2_idx1']),    int(row['itm_top2_idx2']),
        int(row['jina_top2_idx1']),   int(row['jina_top2_idx2']),
        int(row['cosine_top2_idx1']), int(row['cosine_top2_idx2']),
    ]

    vote_counts  = Counter(votes)
    top2_indices = [idx for idx, _ in vote_counts.most_common(2)]

    if len(top2_indices) < 2:
        top2_indices = [int(row['itm_top2_idx1']), int(row['jina_top2_idx1'])]

    voted_results.append({
        "image_name"  : row['image_name'],
        "voted_cap1"  : captions[top2_indices[0]],
        "voted_cap2"  : captions[top2_indices[1]],
        "voted_idx1"  : top2_indices[0],
        "voted_idx2"  : top2_indices[1],
        "vote_counts" : str(dict(vote_counts)),
    })

voted_df = pd.DataFrame(voted_results)
voted_df.to_csv("/kaggle/working/majority_voted_captions.csv", index=False)

print(f" Done  — {len(voted_df)} images voted, {skipped} skipped")
print("\nSample:")
print(voted_df[['image_name','voted_cap1','voted_cap2']].head(3).to_string())

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

IMAGES_PATH = "/content/drive/MyDrive/flickr8k/Images"

samples = voted_df.sample(3, random_state=42)

fig, axes = plt.subplots(3, 2, figsize=(16, 13))
fig.suptitle("Majority Voting — 2 Best Captions per Image",
             fontsize=15, fontweight='bold')

for i, (_, row) in enumerate(samples.iterrows()):

    # Left — actual image
    img_path = os.path.join(IMAGES_PATH, row['image_name'])
    img      = Image.open(img_path).convert("RGB")
    axes[i][0].imshow(img)
    axes[i][0].axis("off")
    axes[i][0].set_title(row['image_name'], fontsize=8, color='gray')

    # Right — voted captions
    axes[i][1].axis("off")

    axes[i][1].text(
        0.05, 0.75,
        f"  {row['voted_cap1']}",
        transform          = axes[i][1].transAxes,
        fontsize           = 11,
        color              = '#1a6e2e',
        fontweight         = 'bold',
        verticalalignment  = 'center',
        wrap               = True,
        bbox               = dict(boxstyle='round,pad=0.6', facecolor='#eafaf1', edgecolor='#27ae60', linewidth=1.5)
    )

    axes[i][1].text(
        0.05, 0.35,
        f" {row['voted_cap2']}",
        transform          = axes[i][1].transAxes,
        fontsize           = 11,
        color              = '#1a3a6e',
        fontweight         = 'bold',
        verticalalignment  = 'center',
        wrap               = True,
        bbox               = dict(boxstyle='round,pad=0.6', facecolor='#eaf4fb', edgecolor='#2980b9', linewidth=1.5)
    )

    axes[i][1].text(
        0.05, 0.08,
        f"Votes: {row['vote_counts']}",
        transform = axes[i][1].transAxes,
        fontsize  = 9,
        color     = 'gray'
    )

plt.tight_layout()
plt.savefig("/kaggle/working/majority_voting_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → /kaggle/working/majority_voting_results.png")

In [ ]:
import pandas as pd
import os

path = "/kaggle/working/majority_voted_captions.csv"
size = os.path.getsize(path) / 1024
df   = pd.read_csv(path)

print(f" File exists  : {path}")
print(f"   Rows         : {len(df)}")
print(f"   Size         : {size:.1f} KB")
print(f"   Columns      : {df.columns.tolist()}")

**Caption fusion**

In [ ]:
# When you need Qwen back

del itm_model
gc.collect()
torch.cuda.empty_cache()

# Then reload Qwen
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct-1M")
model     = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct-1M",
    torch_dtype  = torch.float16,
    device_map   = "auto"
).eval()

In [ ]:
import subprocess
subprocess.run(["pip", "install", "transformers", "accelerate", "-q"])

import os, gc, torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct-1M"

print("Device :", DEVICE)
print("VRAM   :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

# ── Clear everything before loading ──
gc.collect()
torch.cuda.empty_cache()

print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code = True
)

print("Loading Qwen model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype       = torch.float16,
    device_map        = "auto",
    trust_remote_code = True
)
model.eval()

print("\n Qwen loaded!")
print("VRAM used :", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

# ── Sanity test ──
print("\nRunning sanity test...")
test_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "Say hello in one sentence."}
]
test_text   = tokenizer.apply_chat_template(
    test_messages, tokenize=False, add_generation_prompt=True
)
test_inputs = tokenizer(test_text, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    test_out = model.generate(
        **test_inputs,
        max_new_tokens = 20,
        do_sample      = False,
        pad_token_id   = tokenizer.eos_token_id
    )
generated = test_out[0][test_inputs['input_ids'].shape[1]:]
print("Qwen says :", tokenizer.decode(generated, skip_special_tokens=True))
print(" Sanity test passed — ready for fusion")

In [ ]:
import pandas as pd

# ── Paths ──────────────────────────────────────────────────────────
FUSED_PATH = "/content/drive/MyDrive/flickr8k/qwen_fused_captions.csv"

# ── Load ───────────────────────────────────────────────────────────
df = pd.read_csv(FUSED_PATH)

# ── Display 3 examples ─────────────────────────────────────────────
print("=" * 60)
for i, (_, r) in enumerate(df.head(3).iterrows(), 1):
    print(f"\n  Example {i}")
    print(f"  Cap 1  : {r['voted_cap1']}")
    print(f"  Cap 2  : {r['voted_cap2']}")
    print(f"  Fused  : {r['fused_caption']}")
    print("-" * 60)

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────
IMAGES_PATH = "/content/drive/MyDrive/flickr8k/Images"
FUSED_PATH  = "/content/drive/MyDrive/flickr8k/qwen_fused_captions.csv"
SAVE_PATH   = "/content/drive/MyDrive/flickr8k/qwen_fusion_visual.png"

# ── Load ───────────────────────────────────────────────────────────
fused_df = pd.read_csv(FUSED_PATH)
samples  = fused_df.sample(3, random_state=42)

# ── Plot ───────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 16))
fig.suptitle("Qwen2.5 Caption Fusion Results", fontsize=16, fontweight='bold', y=1.01)

for i, (_, row) in enumerate(samples.iterrows()):

    gs      = gridspec.GridSpec(3, 2, figure=fig, hspace=0.6)
    ax_img  = fig.add_subplot(gs[i, 0])
    ax_text = fig.add_subplot(gs[i, 1])

    # ── Left: image ──────────────────────────────────────────────
    img = Image.open(os.path.join(IMAGES_PATH, row['image_name'])).convert("RGB")
    ax_img.imshow(img)
    ax_img.axis("off")
    ax_img.set_title(row['image_name'], fontsize=8, color='gray')

    # ── Right: captions ──────────────────────────────────────────
    ax_text.axis("off")

    # Cap 1
    ax_text.text(
        0.02, 0.92, " Caption 1 (Voted):",
        transform=ax_text.transAxes, fontsize=9,
        color='#2471a3', fontweight='bold'
    )
    ax_text.text(
        0.02, 0.78, row['voted_cap1'],
        transform=ax_text.transAxes, fontsize=9,
        color='#1a1a1a', wrap=True,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#eaf4fb',
                  edgecolor='#2471a3', linewidth=1)
    )

    # Cap 2
    ax_text.text(
        0.02, 0.58, " Caption 2 (Voted):",
        transform=ax_text.transAxes, fontsize=9,
        color='#7d3c98', fontweight='bold'
    )
    ax_text.text(
        0.02, 0.44, row['voted_cap2'],
        transform=ax_text.transAxes, fontsize=9,
        color='#1a1a1a', wrap=True,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#f5eef8',
                  edgecolor='#7d3c98', linewidth=1)
    )

    # Fused
    ax_text.text(
        0.02, 0.26, " Qwen Fused Caption:",
        transform=ax_text.transAxes, fontsize=9,
        color='#1e8449', fontweight='bold'
    )
    ax_text.text(
        0.02, 0.10, row['fused_caption'],
        transform=ax_text.transAxes, fontsize=9,
        color='#1a1a1a', wrap=True,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#eafaf1',
                  edgecolor='#1e8449', linewidth=1.5)
    )

plt.savefig(SAVE_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {SAVE_PATH}")

In [ ]:
# Load captions
df = pd.read_csv('/content/drive/MyDrive/flickr8k/qwen_fused_captions.csv')
IMAGE_DIR = '/content/drive/MyDrive/flickr8k/Images'

# Pick 3 samples that have images
samples = []
for _, row in df.iterrows():
    img_col = df.columns[0]  # first column = image filename
    cap_col = df.columns[1]  # second column = caption
    fname = str(row[img_col])
    path = os.path.join(IMAGE_DIR, fname)
    if os.path.exists(path):
        samples.append((path, str(row[cap_col])))
    if len(samples) == 3:
        break

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.patch.set_facecolor('white')

for ax, (img_path, caption) in zip(axes, samples):
    img = mpimg.imread(img_path)
    ax.imshow(img)
    ax.axis('off')
    # Wrap caption at ~60 chars
    words = caption.split()
    lines, line = [], []
    for w in words:
        line.append(w)
        if len(' '.join(line)) > 60:
            lines.append(' '.join(line[:-1]))
            line = [w]
    lines.append(' '.join(line))
    wrapped = '\n'.join(lines)
    ax.set_title(wrapped, fontsize=11, fontfamily='serif',
                 color='#1A1A1A', pad=10, wrap=True,
                 ha='center', va='top')

plt.tight_layout(pad=1.5)
plt.savefig('figure_1_1_captioning_examples.png',
            dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("Saved as figure_1_1_captioning_examples.png")

**Evaluation Metrices**

In [ ]:
import subprocess
subprocess.run(["pip", "install", "nltk", "rouge-score", "-q"])

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')
nltk.download('punkt_tab')

import pandas as pd
import numpy as np
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
import warnings
warnings.filterwarnings('ignore')

# ── Load fused captions ──
fused_df = pd.read_csv("/content/drive/MyDrive/flickr8k/qwen_fused_captions.csv")

# ── Load original 5 human captions ──
CAPTIONS_PATH = "/content/drive/MyDrive/flickr8k/Flickr_TextData/Flickr8k.token.txt"

captions_dict = {}
with open(CAPTIONS_PATH, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split('\t')
        if len(parts) == 2:
            img_id, caption = parts
            img_name = img_id.split('#')[0]
            if img_name not in captions_dict:
                captions_dict[img_name] = []
            captions_dict[img_name].append(caption.lower().strip())

print(f" Fused captions loaded  : {len(fused_df)}")
print(f" Human captions loaded  : {len(captions_dict)} images")
print(f"\nSample fused caption:")
print(f"  {fused_df['fused_caption'].iloc[0]}")
print(f"\nSample human references:")
for c in captions_dict[fused_df['image_name'].iloc[0]]:
    print(f"  → {c}")

In [ ]:
# BLEU Score
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import matplotlib.pyplot as plt

smoother = SmoothingFunction().method1

refs_corpus  = []   # list of list of reference token lists
hyps_corpus  = []   # list of hypothesis token lists

skipped = 0
for _, row in fused_df.iterrows():
    img   = row['image_name']
    fused = str(row['fused_caption']).lower().strip()

    if img not in captions_dict or not fused:
        skipped += 1
        continue

    refs  = [cap.split() for cap in captions_dict[img]]   # 5 references
    hyp   = fused.split()

    refs_corpus.append(refs)
    hyps_corpus.append(hyp)

# ── Corpus BLEU ──
bleu1 = corpus_bleu(refs_corpus, hyps_corpus, weights=(1, 0, 0, 0))
bleu2 = corpus_bleu(refs_corpus, hyps_corpus, weights=(0.5, 0.5, 0, 0))
bleu3 = corpus_bleu(refs_corpus, hyps_corpus, weights=(0.33, 0.33, 0.33, 0))
bleu4 = corpus_bleu(refs_corpus, hyps_corpus, weights=(0.25, 0.25, 0.25, 0.25))

print("=" * 45)
print("         BLEU SCORES — Fused Captions")
print("=" * 45)
print(f"  BLEU-1  :  {bleu1:.4f}  ({bleu1*100:.2f}%)")
print(f"  BLEU-2  :  {bleu2:.4f}  ({bleu2*100:.2f}%)")
print(f"  BLEU-3  :  {bleu3:.4f}  ({bleu3*100:.2f}%)")
print(f"  BLEU-4  :  {bleu4:.4f}  ({bleu4*100:.2f}%)")
print("=" * 45)
print(f"  Images evaluated : {len(hyps_corpus)}")
print(f"  Skipped          : {skipped}")
print()
print("  Interpretation:")
print(f"  BLEU-1 measures unigram overlap (basic word match)")
print(f"  BLEU-4 measures 4-gram fluency (sentence quality)")

# ── Bar chart ──
fig, ax = plt.subplots(figsize=(8, 5))
metrics = ['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4']
scores  = [bleu1, bleu2, bleu3, bleu4]
colors  = ['#2ecc71', '#3498db', '#9b59b6', '#e67e22']

bars = ax.bar(metrics, scores, color=colors, edgecolor='white', width=0.5)
ax.set_ylim(0, 1)
ax.set_title("BLEU Scores — Qwen Fused Captions vs Human References",
             fontsize=12, fontweight='bold')
ax.set_ylabel("Score")
ax.axhline(0.5, color='red', linestyle='--', linewidth=0.8, label='0.5 reference line')

for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f"{score:.4f}", ha='center', fontsize=11, fontweight='bold')

ax.legend()
plt.tight_layout()
plt.savefig("/kaggle/working/bleu_scores.png", dpi=150, bbox_inches="tight")
plt.show()
print(" Saved → /kaggle/working/bleu_scores.png")

In [ ]:
# METEOR Score

meteor_scores = []
skipped       = 0

for _, row in fused_df.iterrows():
    img   = row['image_name']
    fused = str(row['fused_caption']).lower().strip()

    if img not in captions_dict or not fused:
        skipped += 1
        continue

    refs = [cap.split() for cap in captions_dict[img]]
    hyp  = fused.split()

    # METEOR takes best score across all references
    score = max(meteor_score([ref], hyp) for ref in refs)
    meteor_scores.append(score)

avg_meteor = np.mean(meteor_scores)
med_meteor = np.median(meteor_scores)

print("=" * 45)
print("       METEOR SCORE — Fused Captions")
print("=" * 45)
print(f"  Mean METEOR  :  {avg_meteor:.4f}  ({avg_meteor*100:.2f}%)")
print(f"  Median       :  {med_meteor:.4f}  ({med_meteor*100:.2f}%)")
print(f"  Min          :  {min(meteor_scores):.4f}")
print(f"  Max          :  {max(meteor_scores):.4f}")
print("=" * 45)
print(f"  Images evaluated : {len(meteor_scores)}")
print(f"  Skipped          : {skipped}")
print()
print("  Interpretation:")
print("  METEOR accounts for synonyms, stemming & word order")
print("  Score > 0.25 is considered good for image captioning")

# ── Distribution plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(meteor_scores, bins=40, color='#3498db', edgecolor='white')
axes[0].axvline(avg_meteor, color='red',    linestyle='--', label=f'Mean: {avg_meteor:.4f}')
axes[0].axvline(med_meteor, color='orange', linestyle='--', label=f'Median: {med_meteor:.4f}')
axes[0].set_title("METEOR Score Distribution", fontsize=12, fontweight='bold')
axes[0].set_xlabel("METEOR Score")
axes[0].set_ylabel("Number of Images")
axes[0].legend()

# Summary bar
axes[1].bar(['Mean METEOR', 'Median METEOR'],
            [avg_meteor, med_meteor],
            color=['#3498db', '#2ecc71'], edgecolor='white', width=0.4)
axes[1].set_ylim(0, 1)
axes[1].set_title("METEOR Summary", fontsize=12, fontweight='bold')
axes[1].set_ylabel("Score")
for i, v in enumerate([avg_meteor, med_meteor]):
    axes[1].text(i, v + 0.01, f"{v:.4f}", ha='center',
                 fontsize=12, fontweight='bold')

plt.suptitle("METEOR Score — Qwen Fused Captions vs Human References",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("/kaggle/working/meteor_scores.png", dpi=150, bbox_inches="tight")
plt.show()
print(" Saved → /kaggle/working/meteor_scores.png")

In [ ]:
print("=" * 50)
print("     EVALUATION RESULTS — Qwen Fused Captions")
print("=" * 50)

print("""
  BLEU Scores:
  ┌─────────┬────────┬──────────────────────────────┐
  │ BLEU-1  │ 0.6477 │ 64.77% word overlap          │
  │ BLEU-2  │ 0.5357 │ 53.57% 2-word phrase match   │
  │ BLEU-3  │ 0.4400 │ 44.00% 3-word phrase match   │
  │ BLEU-4  │ 0.3525 │ 35.25% 4-word phrase match   │
  └─────────┴────────┴──────────────────────────────┘

   BLEU Result:
     Fused captions share 64% of words with human
     references. Even at 4-gram level (BLEU-4), 35%
     of 4-word phrases exactly match — which means
     Qwen is writing fluent, human-like sentences.

  METEOR Score:
  ┌──────────────┬────────┬────────────────────────┐
  │ Mean METEOR  │ 0.6332 │ 63.32%                 │
  │ Median       │ 0.6422 │ 64.22%                 │
  └──────────────┴────────┴────────────────────────┘

   METEOR Result:
     63% semantic match with human references after
     accounting for synonyms and word variations.
     Threshold for "good" is 0.25 — we are at 0.63,
     which is 2.5x above the good threshold.
""")
print("=" * 50)

In [ ]:
# CIDEr Score

from pycocoevalcap.cider.cider import Cider
import matplotlib.pyplot as plt

# ── Build in correct format — plain strings, not dicts ──
gts = {}
res = {}

for idx, row in fused_df.iterrows():
    img   = row['image_name']
    fused = str(row['fused_caption']).lower().strip()

    if img not in captions_dict or not fused:
        continue

    gts[idx] = [c for c in captions_dict[img]]   # list of plain strings
    res[idx]  = [fused]                            # list with one plain string

# ── Compute CIDEr ──
cider_scorer = Cider()
mean_cider, cider_scores_arr = cider_scorer.compute_score(gts, res)

cider_per_image = list(cider_scores_arr)

print("=" * 45)
print("       CIDEr SCORE — Fused Captions")
print("=" * 45)
print(f"  Mean CIDEr   :  {mean_cider:.4f}")
print(f"  Median       :  {np.median(cider_per_image):.4f}")
print(f"  Min          :  {min(cider_per_image):.4f}")
print(f"  Max          :  {max(cider_per_image):.4f}")
print("=" * 45)
print(f"  Images evaluated : {len(cider_per_image)}")
print()
print("  Interpretation:")
print("  CIDEr > 0.8 is competitive for image captioning")
print("  CIDEr > 1.0 is strong — human-level on some benchmarks")

# ── Distribution plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(cider_per_image, bins=40, color='#e67e22', edgecolor='white')
axes[0].axvline(mean_cider,                   color='red',  linestyle='--',
                label=f'Mean: {mean_cider:.4f}')
axes[0].axvline(np.median(cider_per_image),   color='blue', linestyle='--',
                label=f'Median: {np.median(cider_per_image):.4f}')
axes[0].set_title("CIDEr Score Distribution", fontsize=12, fontweight='bold')
axes[0].set_xlabel("CIDEr Score")
axes[0].set_ylabel("Number of Images")
axes[0].legend()

axes[1].bar(['Mean CIDEr', 'Median CIDEr'],
            [mean_cider, np.median(cider_per_image)],
            color=['#e67e22', '#f39c12'], edgecolor='white', width=0.4)
axes[1].set_title("CIDEr Summary", fontsize=12, fontweight='bold')
axes[1].set_ylabel("Score")
for i, v in enumerate([mean_cider, np.median(cider_per_image)]):
    axes[1].text(i, v + 0.01, f"{v:.4f}", ha='center',
                 fontsize=12, fontweight='bold')

plt.suptitle("CIDEr Score — Qwen Fused Captions vs Human References",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("/kaggle/working/cider_scores.png", dpi=150, bbox_inches="tight")
plt.show()
print(" Saved → /kaggle/working/cider_scores.png")

In [ ]:
'''import subprocess
subprocess.run(["pip", "uninstall", "pycocoevalcap", "-y", "-q"])
subprocess.run(["pip", "install", "git+https://github.com/salaniz/pycocoevalcap.git", "-q"])'''

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from pycocoevalcap.cider.cider import Cider
import random
import ast, re
import nltk
nltk.download('wordnet')

# ── Load data ──
fused_df  = pd.read_csv("/content/drive/MyDrive/flickr8k/qwen_fused_captions.csv")
cosine_df = pd.read_csv("/content/drive/MyDrive/flickr8k/ranking scores/cosine_scores.csv")

smoother  = SmoothingFunction().method1

# ── Build 3 hypothesis sets ──
random_hyps  = []   # Baseline 1: random single caption
cosine_hyps  = []   # Baseline 2: best cosine caption
fused_hyps   = []   # Ours: Qwen fused caption
all_refs     = []   # same references for all 3

# CIDEr format
gts_random  = {}
gts_cosine  = {}
gts_fused   = {}
res_random  = {}
res_cosine  = {}
res_fused   = {}

def parse_captions(val):
    matches = re.findall(r"np\\.str_\('(.*?)'\)", str(val))
    if matches:
        return matches
    try:
        return ast.literal_eval(val)
    except:
        return []

for idx, row in fused_df.iterrows():
    img   = row['image_name']
    fused = str(row['fused_caption']).lower().strip()

    if img not in captions_dict or not fused:
        continue

    refs      = captions_dict[img]                    # 5 human captions
    refs_tok  = [r.split() for r in refs]

    # Random caption
    rand_cap  = random.choice(refs).lower().strip()

    # Best cosine caption
    cos_row   = cosine_df[cosine_df['image_name'] == img]
    if cos_row.empty:
        continue
    cos_cap   = str(cos_row.iloc[0]['cosine_top2_cap1']).lower().strip()

    # Append
    all_refs.append(refs_tok)
    random_hyps.append(rand_cap.split())
    cosine_hyps.append(cos_cap.split())
    fused_hyps.append(fused.split())

    # CIDEr
    gts_random[idx] = refs
    gts_cosine[idx] = refs
    gts_fused[idx]  = refs
    res_random[idx]  = [rand_cap]
    res_cosine[idx]  = [cos_cap]
    res_fused[idx]   = [fused]

print(f" Built comparison sets ─ {len(fused_hyps)} images")

# ── BLEU ──
def get_bleu(refs, hyps):
    return {
        'BLEU-1': corpus_bleu(refs, hyps, weights=(1,0,0,0)),
        'BLEU-2': corpus_bleu(refs, hyps, weights=(0.5,0.5,0,0)),
        'BLEU-3': corpus_bleu(refs, hyps, weights=(0.33,0.33,0.33,0)),
        'BLEU-4': corpus_bleu(refs, hyps, weights=(0.25,0.25,0.25,0.25)),
    }

bleu_random = get_bleu(all_refs, random_hyps)
bleu_cosine = get_bleu(all_refs, cosine_hyps)
bleu_fused  = get_bleu(all_refs, fused_hyps)

# ── METEOR ──
def get_meteor(refs, hyps):
    scores = []
    for ref_list, hyp in zip(all_refs, hyps):
        score = max(meteor_score([r], hyp) for r in ref_list)
        scores.append(score)
    return np.mean(scores)

meteor_random = get_meteor(all_refs, random_hyps)
meteor_cosine = get_meteor(all_refs, cosine_hyps)
meteor_fused  = get_meteor(all_refs, fused_hyps)

# ── CIDEr ──
cider_scorer = Cider()
mean_random, _ = cider_scorer.compute_score(gts_random, res_random)
mean_cosine, _ = cider_scorer.compute_score(gts_cosine, res_cosine)
mean_fused,  _ = cider_scorer.compute_score(gts_fused,  res_fused)

# ── Print comparison table ──
print("\n")
print("=" * 68)
print("         BASELINE COMPARISON ─ All Methods vs Fused Caption")
print("=" * 68)
print(f"  {'Metric':<12} {'Random Caption':>16} {'Best Cosine':>16} {'Qwen Fused':>16}")
print(f"  {'-'*12} {'-'*16} {'-'*16} {'-'*16}")
for k in ['BLEU-1','BLEU-2','BLEU-3','BLEU-4']:
    r = bleu_random[k]
    c = bleu_cosine[k]
    f = bleu_fused[k]
    winner = "  " if f == max(r,c,f) else ""
    print(f"  {k:<12} {r:>16.4f} {c:>16.4f} {f:>16.4f}{winner}")
print(f"  {'METEOR':<12} {meteor_random:>16.4f} {meteor_cosine:>16.4f} {meteor_fused:>16.4f}  {'\u2705' if meteor_fused == max(meteor_random, meteor_cosine, meteor_fused) else ''}")
print(f"  {'CIDEr':<12} {mean_random:>16.4f} {mean_cosine:>16.4f} {mean_fused:>16.4f}  {'\u2705' if mean_fused == max(mean_random, mean_cosine, mean_fused) else ''}")
print("=" * 68)

# ── SEPARATING METRICS FOR COMPACT GRAPHING ──
metrics_pct  = ['BLEU-1','BLEU-2','BLEU-3','BLEU-4','METEOR']
rand_pct     = [bleu_random[m] if 'BLEU' in m else meteor_random for m in metrics_pct]
cos_pct      = [bleu_cosine[m] if 'BLEU' in m else meteor_cosine for m in metrics_pct]
fused_pct    = [bleu_fused[m]  if 'BLEU' in m else meteor_fused  for m in metrics_pct]

# Append CIDEr index values at the end position
metrics_all  = metrics_pct + ['CIDEr']
rand_all     = rand_pct + [mean_random]
cos_all      = cos_pct + [mean_cosine]
fused_all    = fused_pct + [mean_fused]

x     = np.arange(len(metrics_all))
width = 0.25

# Setup plot window bounding canvas parameters cleanly
fig, ax1 = plt.subplots(figsize=(11, 5.5))

# 1. Plot percentage-based trends on Left Axis (ax1)
b1 = ax1.bar(x[:-1] - width, rand_pct, width, label='Random Caption', color='#e74c3c', edgecolor='white')
b2 = ax1.bar(x[:-1],         cos_pct,  width, label='Best Cosine',    color='#3498db', edgecolor='white')
b3 = ax1.bar(x[:-1] + width, fused_pct, width, label='Qwen Fused',     color='#2ecc71', edgecolor='white')

ax1.set_ylabel("Score (BLEU / METEOR)", fontsize=11, fontweight='bold')
ax1.set_ylim(0, 1.1)  # Clear ceiling buffer for text labels

# 2. Instantiate a separate shared x-axis coordinate space for CIDer metric
ax2 = ax1.twinx()
b1_c = ax2.bar(x[-1] - width, [mean_random], width, color='#e74c3c', edgecolor='white')
b2_c = ax2.bar(x[-1],         [mean_cosine], width, color='#3498db', edgecolor='white')
b3_c = ax2.bar(x[-1] + width, [mean_fused],  width, color='#2ecc71', edgecolor='white')

ax2.set_ylabel("Score (CIDEr Scale)", fontsize=11, color='#7f8c8d', fontweight='bold')
ax2.set_ylim(0, max(mean_random, mean_cosine, mean_fused) * 1.15)  # Auto-allocates correct headroom spacing

# Global layout settings
ax1.set_xticks(x)
ax1.set_xticklabels(metrics_all, fontsize=11, fontweight='bold')
ax1.set_title("Baseline Comparison — Random vs Best Cosine vs Qwen Fused", fontsize=12, fontweight='bold', pad=15)

# Merging legends cleanly into a single box container context
lines, labels = ax1.get_legend_handles_labels()
ax1.legend(lines, labels, loc='upper right', fontsize=10)

# Apply text decorators precisely inside structural bounds without overflowing clipping thresholds
def annotate_bars(bars, axis_context):
    for rect in bars:
        h = rect.get_height()
        axis_context.text(rect.get_x() + rect.get_width()/2, h + (axis_context.get_ylim()[1] * 0.01),
                          f"{h:.3f}", ha='center', va='bottom', fontsize=8, fontweight='bold')

# Run annotations against their distinct vertical transformation matrices
annotate_bars(b1, ax1)
annotate_bars(b2, ax1)
annotate_bars(b3, ax1)
annotate_bars(b1_c, ax2)
annotate_bars(b2_c, ax2)
annotate_bars(b3_c, ax2)

plt.tight_layout()
plt.show()